# 02 · Machine Learning Fundamentals

In plain English, machine learning is the art of **letting a computer figure out the rules from examples instead of us writing the rules by hand**. Normally you'd program "if the email contains the word 'lottery', mark it as spam." With machine learning you instead show the computer thousands of emails already labeled spam / not-spam, and it *learns the pattern itself*. This notebook walks you through the core ideas behind that — using tiny datasets, free tools, and zero scary math — so that when we start fine-tuning real language models, every word ("loss", "overfitting", "learning rate") already feels familiar.

## What you'll learn

- **What machine learning actually is**: learning patterns from data instead of hand-coding rules.
- **Supervised vs unsupervised** learning, and where fine-tuning fits.
- **Classification vs regression** — predicting a *category* vs a *number*.
- **Features vs labels**: the inputs vs the answer we want.
- The **train / validation / test split**, and *why* we deliberately hide some data from the model.
- **Overfitting vs underfitting** — with a hands-on demo you can see for yourself.
- What a **loss function** is (a single number measuring how wrong the model is).
- **Gradient descent** intuition: "rolling downhill", and the **learning rate** as step size.
- **Generalization**: doing well on data you've never seen.
- A first **metric** (accuracy), plus a teaser of precision / recall / F1 (coming in notebook 13).

## Why this matters for fine-tuning

Here's the big secret of this whole course: **fine-tuning is just supervised machine learning, applied on top of a model that already knows a lot.**

Every single idea below comes back in every later notebook:

- You'll still split data into **train / validation / test**.
- The model still minimizes a **loss function**.
- It still learns by **gradient descent** with a **learning rate** you choose.
- You'll still fight **overfitting** (memorizing instead of learning).
- You'll still judge success by **generalization** to new examples.

A giant pretrained language model is fancier than the tiny models here, but the *learning process is identical*. Nail these fundamentals and fine-tuning becomes "more of the same, at scale."

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it if you're on Google Colab or a fresh environment that doesn't already have these libraries. Everything here runs comfortably on a plain CPU.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install scikit-learn numpy matplotlib

import numpy as np                      # fast number arrays
import matplotlib.pyplot as plt         # simple plots
from sklearn.datasets import make_classification  # make tiny toy datasets
from sklearn.model_selection import train_test_split  # split data into parts
from sklearn.linear_model import LogisticRegression   # a simple classifier
from sklearn.tree import DecisionTreeClassifier       # used in the overfitting demo
from sklearn.metrics import accuracy_score            # our first metric

np.random.seed(42)   # make randomness repeatable so your numbers match this notebook
print("Imports OK")  # -> Imports OK

## 1. What is machine learning, really?

Think of a child learning to recognize cats. You don't hand them a rulebook ("a cat has whiskers, pointy ears, fur..."). You just point at cats and say "cat" enough times until they get it. Machine learning is the same: we feed the computer many **examples**, and a learning algorithm adjusts its internal numbers until it can predict the right answer on its own.

Two pieces are always involved:

- **Features** — the inputs (what we know). For an email: the words it contains.
- **Label** — the answer we want (what we're predicting). For an email: spam or not-spam.

The model's job is to learn a mapping **features → label**.

In [ ]:
# A toy peek at "features -> label". Each row is one example.
# Features: [hour_of_day, num_links_in_email]   Label: 1 = spam, 0 = not spam
features = [
    [9,  0],   # 9am, no links
    [2,  8],   # 2am, lots of links
    [14, 1],   # 2pm, one link
    [3,  6],   # 3am, several links
]
labels = [0, 1, 0, 1]

for f, y in zip(features, labels):
    kind = "spam" if y == 1 else "not spam"
    print(f"features={f} -> label={y} ({kind})")
# -> features=[9, 0] -> label=0 (not spam)
# -> features=[2, 8] -> label=1 (spam)
# -> ...

**What this does:** It lays out the universal shape of a supervised ML problem — a list of feature rows and a matching list of labels. Notice a pattern the *human* can already spot (late-night + many links tends to be spam). The whole point of ML is to discover that pattern **automatically**.

### ✏️ Exercise

**Task:** Add two more examples to `features` and `labels` that follow the same "late night + many links = spam" idea, then re-run the loop and check the printout matches your intent.

**Hint:** Append a new list like `[1, 9]` to `features` and the matching label `1` to `labels`. Keep the two lists the same length!

In [ ]:
# Sample solution
features = [
    [9,  0],
    [2,  8],
    [14, 1],
    [3,  6],
    [11, 0],   # 11am, no links -> probably not spam
    [1,  9],   # 1am, many links -> probably spam
]
labels = [0, 1, 0, 1, 0, 1]

assert len(features) == len(labels), "Lists must be the same length!"
print("Now have", len(features), "examples")   # -> Now have 6 examples

## 2. Supervised vs unsupervised learning

- **Supervised learning** — you have the **right answers (labels)** for your training examples. You're teaching with an answer key. *(Spam detection, predicting house prices, and — importantly — fine-tuning a language model.)*
- **Unsupervised learning** — you have **no labels**, just raw data, and you ask the computer to find structure on its own (e.g., grouping similar customers together). There's no answer key.

This course lives almost entirely in **supervised** land, because **fine-tuning is supervised**: every training example is a pair of (input, desired output).

In [ ]:
# Supervised: we provide inputs AND the correct outputs (labels).
supervised_example = {"input": "Translate 'hello' to French", "label": "bonjour"}

# Unsupervised: only inputs, no labels. The model finds structure itself.
unsupervised_example = {"input": "a pile of 10,000 unlabeled news articles"}

print("Supervised has a label:", "label" in supervised_example)      # -> True
print("Unsupervised has a label:", "label" in unsupervised_example)  # -> False

**What this does:** It contrasts the two setups with one tiny dict each. The presence (or absence) of a `"label"` key is exactly the difference. Fine-tuning datasets *always* have that label — the desired response.

### ✏️ Exercise

**Task:** Write a one-line function `is_supervised(example)` that returns `True` if the example dict contains a `"label"` key, and test it on both examples above.

**Hint:** A dict membership check `"label" in example` already returns a boolean — just return it.

In [ ]:
# Sample solution
def is_supervised(example):
    return "label" in example

print(is_supervised(supervised_example))    # -> True
print(is_supervised(unsupervised_example))  # -> False

## 3. Classification vs regression

Both are supervised, but they differ in *what kind of answer* they predict:

- **Classification** → predict a **category** from a fixed set. "Spam or not?", "Which of 3 sentiments?", "cat / dog / bird?".
- **Regression** → predict a **number** on a continuous scale. "What price?", "How many days until X?", "What temperature?".

A quick gut check: *if you could meaningfully say the answer is "halfway between two answers," it's regression. If the answers are separate buckets, it's classification.*

In this course we mostly do classification-flavored tasks, but the math underneath language model fine-tuning is closer to "classify the next word out of the whole vocabulary" — a giant classification problem.

In [ ]:
# Same input, two different framings of the prediction task.
review = "The food was great but service was slow."

classification_answer = "mixed"   # a category from {positive, negative, mixed}
regression_answer      = 6.5       # a star rating on a 0-10 scale (a number)

print("Classification predicts a category:", classification_answer)  # -> mixed
print("Regression predicts a number:", regression_answer)            # -> 6.5

**What this does:** It shows the *same* review turned into either a classification target (a bucket) or a regression target (a number). The choice depends on the question you're asking, not the data itself.

### ✏️ Exercise

**Task:** For each task below, label it `"classification"` or `"regression"`: (a) predict tomorrow's temperature, (b) decide if a tumor image is benign or malignant, (c) predict how many minutes a delivery will take.

**Hint:** Ask "is the answer a number on a scale, or one of a few buckets?"

In [ ]:
# Sample solution
answers = {
    "a) tomorrow's temperature": "regression",   # a number
    "b) benign vs malignant":    "classification",  # two buckets
    "c) delivery minutes":       "regression",   # a number
}
for task, kind in answers.items():
    print(task, "->", kind)

## 4. A tiny real dataset

Let's stop hand-typing data and generate a small synthetic 2-class dataset with scikit-learn's `make_classification`. We'll keep it **tiny (120 samples)** and 2-dimensional so we can even draw it.

- `X` will be our **features** — a 120×2 array (120 examples, 2 numbers each).
- `y` will be our **labels** — 120 values, each 0 or 1 (two classes).

In [ ]:
# Make a small, easy-to-visualize dataset: 120 examples, 2 features, 2 classes.
X, y = make_classification(
    n_samples=120,       # 120 examples total (small + CPU-friendly)
    n_features=2,        # 2 features so we can plot them on an x/y grid
    n_redundant=0,       # no duplicate/derived features
    n_informative=2,     # both features actually carry signal
    n_clusters_per_class=1,
    class_sep=1.3,       # how far apart the two classes sit (higher = easier)
    random_state=42,     # repeatable
)

print("X shape:", X.shape)   # -> X shape: (120, 2)   (rows, columns)
print("y shape:", y.shape)   # -> y shape: (120,)
print("first 3 feature rows:\n", X[:3])
print("first 3 labels:", y[:3])
print("class counts:", np.bincount(y))  # -> roughly [60 60]

**What this does:** `make_classification` invents a labeled dataset for us. `X.shape == (120, 2)` means 120 examples each described by 2 numbers; `y` holds the matching 0/1 answers. `np.bincount(y)` confirms the two classes are roughly balanced (~60 each), which keeps accuracy meaningful later.

In [ ]:
# Optional: SEE the data. Color = class. (This is what the model must separate.)
plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=30)
plt.xlabel("feature 1"); plt.ylabel("feature 2")
plt.title("Tiny 2-class dataset")
plt.show()
# Expected: a scatter of ~120 dots in two color groups that mostly separate,
# with a few mixed dots near the boundary.

**What this does:** It plots each example as a dot, colored by its class. You should see two blobs that *mostly* separate but overlap a little near the middle. That little overlap is exactly why no model will ever be 100% perfect — and why we measure accuracy.

### ✏️ Exercise

**Task:** Re-create the dataset with `class_sep=0.5` (classes closer together) and re-plot it. Does it look easier or harder to separate?

**Hint:** Copy the `make_classification` call, change only `class_sep`, store it in `X_hard, y_hard`, then reuse the scatter code.

In [ ]:
# Sample solution
X_hard, y_hard = make_classification(
    n_samples=120, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=0.5, random_state=42,
)
plt.figure(figsize=(5, 4))
plt.scatter(X_hard[:, 0], X_hard[:, 1], c=y_hard, cmap="coolwarm", edgecolor="k", s=30)
plt.title("Harder dataset (class_sep=0.5)")
plt.show()
# The two colors now overlap much more -> harder to separate -> lower accuracy.

## 5. The train / validation / test split (and WHY)

Imagine a teacher who hands out the exact exam questions as homework. Students could *memorize* the answers and ace the test without truly understanding. We'd never know if they actually learned.

Machine learning has the same trap. So we **split our data**:

- **Training set** — the model learns from this (the "homework").
- **Validation set** — used while developing to tune choices and check progress (a "practice exam").
- **Test set** — locked away, used **once** at the very end to estimate real-world performance (the "final exam"). The model must never train on it.

The whole reason we hold data out is to measure **generalization** — performance on examples the model has *never seen*. That number is the only one that matters.

A common split is something like **60% train / 20% validation / 20% test**.

In [ ]:
# Step 1: hold out 20% as the FINAL test set (locked away).
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y  # stratify keeps class balance
)

# Step 2: from what's left, carve out a validation set (20% of the original).
# 0.25 of the remaining 80% = 20% of the whole.
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print("train size:", len(X_train))  # -> 72   (60% of 120)
print("val size:  ", len(X_val))    # -> 24   (20% of 120)
print("test size: ", len(X_test))   # -> 24   (20% of 120)

**What this does:** It splits 120 examples into 72 train / 24 validation / 24 test. We split **twice**: first peel off the test set, then split the rest into train and validation. `stratify=y` keeps the 50/50 class balance in every split so no piece is accidentally lopsided. The test set is now untouched until the very end.

### ✏️ Exercise

**Task:** Print what fraction of the original 120 examples ended up in each split (train, val, test). They should add up to 1.0.

**Hint:** Divide each `len(...)` by `len(X)` and print the three fractions.

In [ ]:
# Sample solution
total = len(X)
print("train fraction:", round(len(X_train) / total, 2))  # -> 0.6
print("val fraction:  ", round(len(X_val) / total, 2))    # -> 0.2
print("test fraction: ", round(len(X_test) / total, 2))   # -> 0.2
print("sum:", round((len(X_train) + len(X_val) + len(X_test)) / total, 2))  # -> 1.0

## 6. Train your first model

`LogisticRegression` is a simple, fast classifier (despite the word "regression" in its name, it does **classification**). We'll:

1. Create the model.
2. `.fit(...)` it on the **training** data — this is the actual "learning".
3. Check its **accuracy** on training data vs the held-out **test** data.

**Accuracy** = (number of correct predictions) / (total predictions). Simple and intuitive for balanced data.

In [ ]:
# Create and TRAIN the model on the training set only.
model = LogisticRegression()
model.fit(X_train, y_train)   # <- this is where learning happens

# Predict on each split and measure accuracy.
train_acc = accuracy_score(y_train, model.predict(X_train))
val_acc   = accuracy_score(y_val,   model.predict(X_val))
test_acc  = accuracy_score(y_test,  model.predict(X_test))

print(f"train accuracy: {train_acc:.2f}")  # -> ~0.93
print(f"val accuracy:   {val_acc:.2f}")    # -> ~0.92
print(f"test accuracy:  {test_acc:.2f}")   # -> ~0.92

**What this does:** `model.fit(X_train, y_train)` adjusts the model's internal numbers to best map features to labels. We then score accuracy on three splits. The key sign of a **healthy** model: train, validation, and test accuracy are all **close together** (here, all around 0.9). That means it learned a real pattern, not noise. (Your exact decimals may differ slightly by scikit-learn version — the *pattern* is what matters.)

### ✏️ Exercise

**Task:** A baseline that always guesses the most common class would score about 0.5 here (classes are 50/50). Compute how many **percentage points** better our model's test accuracy is than that 0.5 baseline.

**Hint:** `(test_acc - 0.5) * 100`.

In [ ]:
# Sample solution
baseline = 0.5
improvement = (test_acc - baseline) * 100
print(f"Model beats the always-guess baseline by {improvement:.0f} percentage points")
# -> something like "Model beats the always-guess baseline by 42 percentage points"

## 7. Overfitting vs underfitting

These are the two ways learning goes wrong:

- **Underfitting** — the model is **too simple** to capture the pattern. It does poorly on *both* train and test. (Like studying so little you fail the homework too.)
- **Overfitting** — the model is **too complex** and **memorizes** the training data, including its random noise. It does great on train but **poorly on test**. (Like memorizing the homework answers word-for-word and then panicking on slightly different exam questions.)

The tell-tale sign of overfitting: **train accuracy ≫ test accuracy** (a big gap).

Let's *cause* overfitting on purpose with a very deep decision tree, which is happy to memorize.

In [ ]:
# A SHALLOW tree (depth 1) is too simple -> likely UNDERFITS.
under = DecisionTreeClassifier(max_depth=1, random_state=42).fit(X_train, y_train)
print("UNDERFIT  train:", f"{accuracy_score(y_train, under.predict(X_train)):.2f}",
      "| test:", f"{accuracy_score(y_test, under.predict(X_test)):.2f}")

# A VERY DEEP tree memorizes the training data -> OVERFITS.
over = DecisionTreeClassifier(max_depth=None, random_state=42).fit(X_train, y_train)
print("OVERFIT   train:", f"{accuracy_score(y_train, over.predict(X_train)):.2f}",
      "| test:", f"{accuracy_score(y_test, over.predict(X_test)):.2f}")

# Our logistic regression from before: a healthy middle ground.
print("BALANCED  train:", f"{train_acc:.2f}", "| test:", f"{test_acc:.2f}")
# Expected pattern:
#   UNDERFIT : train and test BOTH mediocre (e.g. ~0.85 / ~0.83)
#   OVERFIT  : train ~1.00 but test clearly LOWER (a visible gap)
#   BALANCED : train and test close together and good

**What this does:** It trains three models of increasing complexity and prints train-vs-test accuracy for each. Watch the **overfit** tree hit ~1.00 on train while its test score drops — that gap **is** overfitting. The shallow tree underfits (both scores lower). Logistic regression sits in the healthy middle. Exact numbers vary, but the *shape* of the result is the lesson.

In [ ]:
# Make the overfitting gap impossible to miss: print it as a single number.
gap = accuracy_score(y_train, over.predict(X_train)) - accuracy_score(y_test, over.predict(X_test))
print(f"Overfit model's train-minus-test gap: {gap:.2f}")
# A large positive gap (e.g. 0.10-0.20+) = the model memorized instead of generalizing.

**What this does:** It reduces overfitting to one diagnostic number: train accuracy minus test accuracy. A small gap is healthy; a big positive gap is a red flag. **You will watch this exact gap when fine-tuning language models** — training loss dropping while validation loss rises is the same warning sign.

### ✏️ Exercise

**Task:** Train decision trees at `max_depth` of 2, 4, and 8. Print train and test accuracy for each and find the depth where the train-minus-test gap starts to blow up.

**Hint:** Loop over `[2, 4, 8]`, build a `DecisionTreeClassifier(max_depth=d, random_state=42)`, fit, and print both accuracies.

In [ ]:
# Sample solution
for d in [2, 4, 8]:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test,  m.predict(X_test))
    print(f"depth={d}: train={tr:.2f} test={te:.2f} gap={tr - te:.2f}")
# As depth grows, train climbs toward 1.0 while test stalls or dips -> gap widens.

## 8. What is a loss function?

A **loss function** is just **a single number that says how wrong the model currently is.** Lower is better; 0 would mean perfect.

- For regression, a common loss is **mean squared error**: average of (prediction − truth) squared.
- For classification (and language models), it's usually **cross-entropy**, which is large when the model is confidently wrong.

Training = **changing the model's numbers to make the loss smaller.** That's the entire game.

Let's compute a simple loss by hand so it's concrete.

In [ ]:
# Mean Squared Error (MSE): how far off are our number predictions, on average?
true_values = np.array([3.0, 5.0, 2.0, 8.0])
predicted   = np.array([2.5, 5.5, 2.0, 6.0])  # some good, one (8 vs 6) quite off

errors = predicted - true_values     # how wrong each prediction is
mse = np.mean(errors ** 2)           # square (so + and - don't cancel), then average
print("errors:", errors)             # -> [-0.5  0.5  0.  -2. ]
print("MSE (loss):", round(mse, 3))  # -> 1.125  (lower is better)

**What this does:** It turns "how wrong are we?" into one number. We square the errors so that positive and negative misses don't cancel out, and so that big mistakes (the 8-vs-6 one) get punished extra. That single `mse` value is exactly the kind of number training tries to push down.

### ✏️ Exercise

**Task:** Improve the predictions so the loss drops. Change `predicted` to be closer to `true_values` and recompute the MSE — aim for under 0.1.

**Hint:** The worst miss is the last one (predicted 6.0 vs true 8.0). Nudge predictions toward the true values.

In [ ]:
# Sample solution
better = np.array([3.0, 5.0, 2.0, 7.9])   # nearly perfect now
mse_better = np.mean((better - true_values) ** 2)
print("new MSE:", round(mse_better, 3))   # -> 0.0025  (much lower = much better)

## 9. Gradient descent: rolling downhill

How does the model actually make the loss smaller? **Gradient descent.**

Picture the loss as a **hilly landscape** and the model standing somewhere on it. The height is the loss; we want the **lowest valley**. At each step, the model:

1. Looks at which direction is **downhill** (that's the "gradient").
2. Takes a small step that way.
3. Repeats until it can't go much lower.

The size of each step is the **learning rate**:

- **Too small** → painfully slow; takes forever to reach the bottom.
- **Too large** → you overshoot the valley and bounce around, maybe never settling.
- **Just right** → steady, efficient progress.

The learning rate is one of the most important knobs you'll set when fine-tuning. Let's simulate it on a simple bowl-shaped loss.

In [ ]:
# Toy loss: a simple bowl, loss(x) = (x - 3)**2. Its lowest point is at x = 3.
def loss(x):       return (x - 3) ** 2
def gradient(x):   return 2 * (x - 3)   # the slope; points uphill, so we step AGAINST it

x = 0.0                 # start far from the bottom
learning_rate = 0.1     # step size — try changing this!

for step in range(15):
    g = gradient(x)               # which way is uphill, and how steep
    x = x - learning_rate * g     # step downhill (against the gradient)
    if step % 3 == 0:             # print every 3rd step to keep it short
        print(f"step {step:2d}: x={x:.3f}  loss={loss(x):.3f}")

print(f"final x={x:.3f} (true minimum is 3.0)")
# Expected: x marches from 0 toward ~3.0 and loss shrinks toward ~0.

**What this does:** It rolls a point downhill on the bowl `loss(x) = (x-3)^2`. Each step moves `x` against the gradient by `learning_rate` times the slope. Watch `x` climb toward 3 and `loss` fall toward 0. This *is* how neural networks and fine-tuned LLMs learn — just with millions of parameters instead of one `x`.

### ✏️ Exercise

**Task:** Try `learning_rate = 1.1` (too big) and re-run. What happens to `x` and the loss over the steps? Then try `0.01` (too small).

**Hint:** Just change the `learning_rate` value and re-run the loop. With 1.1 the steps overshoot and *grow*; with 0.01 progress is so slow it barely moves in 15 steps.

In [ ]:
# Sample solution: compare a too-big vs a too-small learning rate.
for lr in [1.1, 0.01]:
    x = 0.0
    for _ in range(15):
        x = x - lr * gradient(x)
    print(f"lr={lr}: final x={x:.3f}, loss={loss(x):.3f}")
# lr=1.1  -> x explodes away from 3 (overshoot), loss huge.
# lr=0.01 -> x barely moved from 0 (too slow), loss still large.

## 10. Generalization and a peek at better metrics

**Generalization** is the goal of everything above: a model that does well on **new, unseen** data, not just the data it trained on. The test set is how we estimate it. Overfitting is the enemy of generalization; holding out data is how we catch it.

**Accuracy** got us started, but it can mislead. Imagine a disease that affects 1% of people. A lazy model that *always* predicts "healthy" is **99% accurate** — and completely useless, because it never catches the disease.

For cases like that we need richer metrics:

- **Precision** — of the cases we flagged positive, how many really were?
- **Recall** — of the truly positive cases, how many did we catch?
- **F1** — a single balance of precision and recall.

We'll dive into these properly in **notebook 13**. For now, just remember: *accuracy is a fine first glance, but not the whole story.*

In [ ]:
# Why accuracy alone can lie: an imbalanced example.
# 100 patients, only 1 is actually sick. A model that ALWAYS says "healthy":
truth     = np.array([0]*99 + [1])    # 99 healthy (0), 1 sick (1)
lazy_pred = np.array([0]*100)         # predicts healthy for everyone

print("Accuracy of the 'always healthy' model:",
      accuracy_score(truth, lazy_pred))   # -> 0.99  (looks great!)
print("...but it caught the sick patient? ->",
      bool(lazy_pred[truth == 1][0] == 1))  # -> False  (it missed the only sick one)

**What this does:** It builds a 99-vs-1 imbalanced case where a do-nothing model scores 99% accuracy yet **never** identifies the one sick patient. This is the motivating example for precision/recall/F1 — the metrics you'll use to evaluate fine-tuned models when one outcome matters more than another.

### ✏️ Exercise

**Task:** Build a `smart_pred` that correctly flags the sick patient (index 99) as `1` while keeping everyone else `0`. Confirm it *also* has high accuracy AND catches the sick patient.

**Hint:** Copy `truth` into a new array (e.g. `smart_pred = truth.copy()`), since a perfect predictor here equals the truth.

In [ ]:
# Sample solution
smart_pred = truth.copy()   # a model that gets this one right
print("Accuracy:", accuracy_score(truth, smart_pred))          # -> 1.0
print("Caught the sick patient?",
      bool(smart_pred[truth == 1][0] == 1))                     # -> True

## Common mistakes & how to debug them

| Mistake | What you'll see | Fix |
|---|---|---|
| **Evaluating on training data only** | Suspiciously high accuracy, then a flop in the real world | Always report **test** (held-out) accuracy too |
| **Letting test data leak into training** | Test score looks amazing but production fails | Split *first*; never `.fit()` on test data; do scaling/feature-prep using train stats only |
| **Imbalanced classes + accuracy** | 95%+ accuracy on a model that's actually useless | Check `np.bincount(y)`; use precision/recall/F1 (notebook 13) |
| **Train acc ≫ test acc** | Big train-minus-test gap | You're **overfitting** — use a simpler model, more data, or regularization |
| **Both train & test acc low** | Everything mediocre | You're **underfitting** — use a more capable model or better features |
| **Learning rate too high** | Loss jumps around or grows / becomes `nan` | Lower the learning rate |
| **Learning rate too low** | Loss barely moves; training "stuck" | Raise the learning rate (or train longer) |
| **Mismatched lengths** | `ValueError` about inconsistent samples | Ensure `len(X) == len(y)` and shapes line up |
| **Forgetting `random_state`** | Numbers change every run, can't compare | Set `random_state=42` everywhere for reproducibility |

## Summary

- **Machine learning** learns patterns from **examples** instead of hand-written rules.
- **Supervised** learning uses labeled data (an answer key); **unsupervised** finds structure without labels. **Fine-tuning is supervised.**
- **Classification** predicts a category; **regression** predicts a number.
- **Features** are inputs; the **label** is the answer we want.
- We split data into **train / validation / test** and hold data out to measure **generalization** — performance on unseen examples.
- **Underfitting** = too simple (bad everywhere). **Overfitting** = memorizing (great on train, poor on test → watch the **gap**).
- A **loss function** is one number measuring wrongness; training shrinks it.
- **Gradient descent** rolls the loss downhill; the **learning rate** is the step size (too big overshoots, too small crawls).
- **Accuracy** is a handy first metric, but **precision / recall / F1** (notebook 13) matter when classes are imbalanced.
- Every one of these ideas reappears, unchanged, when we fine-tune real language models.

## What to learn next

Next up: **`03_neural_network_basics.ipynb`**.

You now understand *what* a model learns (a mapping from features to labels) and *how* it learns (shrink a loss via gradient descent). Notebook 03 opens up the **model itself** — neurons, layers, weights, and activation functions — showing how those simple pieces stack into the neural networks that power every modern language model. The loss and gradient-descent intuition you just built carries straight over.

See you there!